# Module 03, Notebook 3: CausalPy Synthetic Control

## Learning Objectives
By completing this notebook, you will:
1. Use `cp.pymc_experiments.SyntheticControl` for a full Bayesian synthetic control analysis
2. Extract and interpret the posterior distribution of donor weights
3. Compute the counterfactual trajectory with 94% HDI at each time period
4. Combine Bayesian uncertainty with permutation p-values
5. Compare the CausalPy result to the manual scipy result from Notebook 01

## Why Bayesian Synthetic Control?

Classical synthetic control gives a point estimate for donor weights and the counterfactual gap.
CausalPy's Bayesian approach gives a **posterior distribution** over both:

- Donor weights: uncertainty in which states contribute most
- Counterfactual trajectory: uncertainty about what California's consumption would have been
- Causal effect at each post-period: credible intervals, not just point estimates

This is especially valuable when the donor pool is small and point estimates are unstable.

## Estimated Time: 15 minutes

---

In [ ]:
learning_objectives(["Use `cp.pymc_experiments.SyntheticControl` for a full Bayesian synthetic control analysis", "Extract and interpret the posterior distribution of donor weights", "Compute the counterfactual trajectory with 94% HDI at each time period", "Combine Bayesian uncertainty with permutation p-values", "Compare the CausalPy result to the manual scipy result from Notebook 01"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import causalpy as cp
from scipy.optimize import minimize

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

In [ ]:
# Recreate the panel dataset and format it for CausalPy
# CausalPy expects long-format data with columns: [group, time, outcome]

N_YEARS = 30
N_PRE = 20
N_POST = 10
N_DONORS = 20
T_STAR = N_PRE
TRUE_LEVEL_EFFECT = -20.0
TRUE_SLOPE_EFFECT = -0.5

years = np.arange(N_YEARS)

np.random.seed(42)
donor_baselines = np.random.uniform(100, 140, N_DONORS)
shared_trend = -1.2
donor_own_trends = np.random.normal(0, 0.3, N_DONORS)

Y_donors = np.zeros((N_YEARS, N_DONORS))
for j in range(N_DONORS):
    trend = shared_trend + donor_own_trends[j]
    Y_donors[:, j] = (
        donor_baselines[j] + trend * years
        + np.random.normal(0, 3.0, N_YEARS)
    )

TRUE_WEIGHTS = np.zeros(N_DONORS)
TRUE_WEIGHTS[0] = 0.40
TRUE_WEIGHTS[3] = 0.35
TRUE_WEIGHTS[7] = 0.25

y_california_latent = Y_donors @ TRUE_WEIGHTS + np.random.normal(0, 1.5, N_YEARS)
y_california = y_california_latent.copy()
for t in range(T_STAR, N_YEARS):
    time_post = t - T_STAR
    y_california[t] += TRUE_LEVEL_EFFECT + TRUE_SLOPE_EFFECT * time_post

# Build long-format DataFrame
state_names = [f"Donor_{j:02d}" for j in range(N_DONORS)]
state_names[0] = "Colorado"
state_names[3] = "Utah"
state_names[7] = "Montana"

rows = []
for t in range(N_YEARS):
    rows.append({"state": "California", "year": t, "cigsale": y_california[t]})
for j, name in enumerate(state_names):
    for t in range(N_YEARS):
        rows.append({"state": name, "year": t, "cigsale": Y_donors[t, j]})

panel_df = pd.DataFrame(rows)
print(f"Panel shape: {panel_df.shape}")
print(f"Unique states: {panel_df['state'].nunique()}")
print(panel_df.head(6))

## 1. Fit the CausalPy Synthetic Control Model

CausalPy's `SyntheticControl` uses a Bayesian approach: a Dirichlet prior on the donor weights
ensures non-negativity and sum-to-one constraints. The NUTS sampler estimates the full posterior
distribution over weights and counterfactual outcomes.

In [ ]:
# Fit Bayesian synthetic control using CausalPy
# WeightedSumFitter places a Dirichlet prior on donor weights and uses NUTS

sc_model = cp.pymc_experiments.SyntheticControl(
    data=panel_df,
    treatment_time=T_STAR,
    formula="cigsale ~ 0 + Colorado + Utah + Montana + " +
            " + ".join([f"Donor_{j:02d}" for j in range(N_DONORS)
                        if j not in [0, 3, 7]]),
    group_variable_name="state",
    treated_group="California",
    model=cp.pymc_models.WeightedSumFitter(
        sample_kwargs={
            "draws": 1000,
            "tune": 1000,
            "chains": 4,
            "progressbar": True,
            "random_seed": 42,
        }
    ),
)

print("SyntheticControl model fitted.")

In [ ]:
# Check convergence
summary = az.summary(sc_model.idata, var_names=["beta"], round_to=3)
print("Convergence Summary (donor weights)")
print(summary[["mean", "sd", "hdi_3%", "hdi_97%", "r_hat"]].head(10).to_string())
print()

max_rhat = summary["r_hat"].max()
print(f"Maximum R-hat: {max_rhat:.4f}  (should be < 1.01)")
if max_rhat < 1.01:
    print("  CONVERGED: All R-hat values below 1.01.")
else:
    print("  WARNING: Some parameters have not converged. Increase tune/draws.")

## 2. Posterior Weight Distribution

The Bayesian approach gives a posterior distribution over donor weights, not just a point estimate.
Visualize the weight posterior for each donor state.

In [ ]:
# Extract weight posterior: variable 'beta' in CausalPy WeightedSumFitter
# Shape: (chains, draws, n_donors)
beta_posterior = sc_model.idata.posterior["beta"].values
beta_flat = beta_posterior.reshape(-1, N_DONORS)  # (n_samples, n_donors)

# Posterior mean weights
weight_means = beta_flat.mean(axis=0)
weight_hdi = az.hdi(beta_flat, hdi_prob=0.94)

# Sort by posterior mean
sort_idx = np.argsort(weight_means)[::-1]

# Display top weights
all_state_names = ["Colorado", "Utah", "Montana"] + \
                  [f"Donor_{j:02d}" for j in range(N_DONORS) if j not in [0, 3, 7]]

print("Posterior Donor Weights (sorted by posterior mean)")
print(f"{'State':<15} {'Mean':>8} {'HDI Low':>10} {'HDI High':>10} {'True':>8}")
print("-" * 55)
for rank, idx in enumerate(sort_idx[:8]):  # show top 8
    true_w = TRUE_WEIGHTS[0] if idx == 0 else (
             TRUE_WEIGHTS[3] if idx == 3 else (
             TRUE_WEIGHTS[7] if idx == 7 else 0.0))
    name = all_state_names[idx] if idx < len(all_state_names) else f"Donor_{idx:02d}"
    print(f"  {name:<13} {weight_means[idx]:>8.3f} {weight_hdi[idx, 0]:>10.3f} "
          f"{weight_hdi[idx, 1]:>10.3f} {true_w:>8.3f}")

print()
print(f"Sum of posterior mean weights: {weight_means.sum():.6f} (should be ~1.0)")

In [ ]:
# Visualize weight posterior for top donors

n_top = 6
top_idx = sort_idx[:n_top]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for rank, (ax, idx) in enumerate(zip(axes, top_idx)):
    samples = beta_flat[:, idx]
    ax.hist(samples, bins=50, color="#3498db", edgecolor="white", alpha=0.8, density=True)
    ax.axvline(weight_means[idx], color="#2980b9", linewidth=2,
               label=f"Mean: {weight_means[idx]:.3f}")

    # True weight (if applicable)
    true_w = TRUE_WEIGHTS[0] if idx == 0 else (
             TRUE_WEIGHTS[3] if idx == 3 else (
             TRUE_WEIGHTS[7] if idx == 7 else 0.0))
    if true_w > 0:
        ax.axvline(true_w, color="#27ae60", linewidth=2, linestyle=":",
                   label=f"True: {true_w:.2f}")

    name = all_state_names[idx] if idx < len(all_state_names) else f"Donor_{idx:02d}"
    ax.set_title(f"{name}", fontsize=11)
    ax.set_xlabel("Weight")
    ax.legend(fontsize=9)

plt.suptitle("Posterior Distribution of Donor Weights (Top 6 States)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Key donors (Colorado, Utah, Montana) should have clearly positive weight posteriors.")
print("Their true weights (0.40, 0.35, 0.25) should fall within the HDI.")

## 3. Counterfactual Trajectory with Uncertainty

The Bayesian approach gives a posterior distribution over the counterfactual at each time point.
Visualize the treated unit, the synthetic counterfactual with 94% HDI, and the causal effect gap.

In [ ]:
# Compute posterior predictive counterfactual
# For each posterior weight sample, compute the synthetic California

# Y_donors at all time points: shape (N_YEARS, N_DONORS)
# Counterfactual: Y_donors @ weights at each time point
# Shape: (n_samples, N_YEARS)

counterfactual_samples = beta_flat @ Y_donors.T  # (n_samples, N_YEARS)

# Posterior summaries
cf_mean = counterfactual_samples.mean(axis=0)
cf_lower = np.percentile(counterfactual_samples, 3, axis=0)
cf_upper = np.percentile(counterfactual_samples, 97, axis=0)

# Gap: treated - counterfactual
gap_samples = y_california[np.newaxis, :] - counterfactual_samples  # (n_samples, N_YEARS)
gap_mean = gap_samples.mean(axis=0)
gap_lower = np.percentile(gap_samples, 3, axis=0)
gap_upper = np.percentile(gap_samples, 97, axis=0)

# --- Main trajectory plot ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.fill_between(years, cf_lower, cf_upper, alpha=0.2, color="#e74c3c", label="94% HDI (counterfactual)")
ax.plot(years, cf_mean, color="#e74c3c", linewidth=2, linestyle="--", label="Synthetic California (mean)")
ax.plot(years, y_california, color="#2980b9", linewidth=2.5, label="California (observed)")
ax.axvline(T_STAR, color="#27ae60", linestyle=":", linewidth=2, label="Tobacco tax")
ax.set_title("California vs Bayesian Synthetic California", fontsize=12)
ax.set_xlabel("Year")
ax.set_ylabel("Packs per Capita per Year")
ax.legend(fontsize=10)

ax = axes[1]
ax.fill_between(years[N_PRE:], gap_lower[N_PRE:], gap_upper[N_PRE:],
                alpha=0.3, color="#e74c3c", label="94% HDI (gap)")
ax.plot(years[:N_PRE], gap_mean[:N_PRE], color="#aaaaaa", linewidth=1.5, label="Pre-period gap")
ax.plot(years[N_PRE:], gap_mean[N_PRE:], color="#e74c3c", linewidth=2.5, label="Causal effect (mean)")

# True effect
true_effects = np.array([TRUE_LEVEL_EFFECT + TRUE_SLOPE_EFFECT * t for t in range(N_POST)])
ax.plot(years[N_PRE:], true_effects, color="#27ae60", linewidth=2, linestyle=":",
        label="True effect")

ax.axhline(0, color="black", linestyle="--", linewidth=1.5)
ax.axvline(T_STAR, color="#27ae60", linestyle=":", linewidth=2)
ax.set_title("Causal Effect: Bayesian SC Gap with 94% HDI", fontsize=12)
ax.set_xlabel("Year")
ax.set_ylabel("Gap (packs per capita)")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 4. Summary Table: Pointwise Causal Effects

In [ ]:
# Pointwise causal effect table with HDI

gap_post_samples = gap_samples[:, N_PRE:]  # (n_samples, N_POST)
gap_hdi_post = az.hdi(gap_post_samples, hdi_prob=0.94)

print("Pointwise Causal Effect Estimates (Bayesian SC)")
print("=" * 65)
print(f"{'Year':<6} {'California':>12} {'Synthetic':>12} {'Gap Mean':>10} "
      f"{'HDI Low':>10} {'HDI High':>10} {'True':>8}")
print("-" * 65)

for i, year in enumerate(years[N_PRE:]):
    true_eff = TRUE_LEVEL_EFFECT + TRUE_SLOPE_EFFECT * i
    print(f"  {year:<4}  {y_california[N_PRE + i]:>12.1f}  {cf_mean[N_PRE + i]:>12.1f}  "
          f"{gap_mean[N_PRE + i]:>+10.1f}  "
          f"{gap_hdi_post[i, 0]:>+10.1f}  {gap_hdi_post[i, 1]:>+10.1f}  "
          f"{true_eff:>+8.1f}")

print("-" * 65)
cum_effect_samples = gap_post_samples.sum(axis=1)
cum_hdi = az.hdi(cum_effect_samples, hdi_prob=0.94)
true_cum = sum(TRUE_LEVEL_EFFECT + TRUE_SLOPE_EFFECT * i for i in range(N_POST))
print(f"  Cumulative effect: {cum_effect_samples.mean():>+10.1f}  "
      f"[{cum_hdi[0]:>+8.1f}, {cum_hdi[1]:>+8.1f}]  True: {true_cum:>+8.1f}")

print()
# Check if zero is excluded from cumulative HDI
if cum_hdi[0] > 0 or cum_hdi[1] < 0:
    print("The 94% HDI for the cumulative effect excludes zero — strong evidence of an effect.")
else:
    print("The 94% HDI for the cumulative effect includes zero — uncertain evidence.")

## 5. Compare: Classical (scipy) vs Bayesian (CausalPy)

Both methods should give similar point estimates. The Bayesian approach adds credible intervals
that the classical method lacks.

In [ ]:
# Classical synthetic control (from Notebook 01)
def sc_weights_classical(y_treated_pre, Y_donors_pre):
    n_donors = Y_donors_pre.shape[1]
    w0 = np.ones(n_donors) / n_donors
    def obj(w):
        return np.sum((y_treated_pre - Y_donors_pre @ w) ** 2)
    result = minimize(
        obj, w0, method="SLSQP",
        bounds=[(0.0, 1.0)] * n_donors,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
        options={"maxiter": 2000, "ftol": 1e-12},
    )
    return result.x


w_classical = sc_weights_classical(y_california[:N_PRE], Y_donors[:N_PRE, :])
y_syn_classical_post = Y_donors[N_PRE:, :] @ w_classical
gap_classical = y_california[N_PRE:] - y_syn_classical_post

print("Comparison: Classical SC vs Bayesian SC")
print(f"{'Year':<6} {'Classical Gap':>16} {'Bayesian Gap Mean':>20} {'Bayesian HDI':>25} {'True':>8}")
print("-" * 78)
for i, year in enumerate(years[N_PRE:]):
    true_eff = TRUE_LEVEL_EFFECT + TRUE_SLOPE_EFFECT * i
    print(f"  {year:<4}  {gap_classical[i]:>+14.1f}  {gap_mean[N_PRE + i]:>+18.1f}  "
          f"[{gap_hdi_post[i, 0]:>+7.1f}, {gap_hdi_post[i, 1]:>+7.1f}]  "
          f"{true_eff:>+8.1f}")

print()
print("Key insight: Both methods give similar point estimates.")
print("Bayesian SC adds credible intervals — crucial for reporting uncertainty.")

## Summary

### CausalPy Synthetic Control vs Manual scipy

| Aspect | Manual scipy | CausalPy (Bayesian) |
|--------|-------------|---------------------|
| Point estimate | Yes | Yes (posterior mean) |
| Uncertainty on weights | No | Full posterior |
| Uncertainty on gap | No (bootstrap only) | Full posterior HDI |
| Convergence diagnostics | No | R-hat, ESS via ArviZ |
| Integration with ArviZ | No | Full InferenceData |

### Complete Reporting Workflow

A complete synthetic control analysis report should include:

1. Pre-period fit plot (treated vs synthetic trajectory)
2. Donor weight table (posterior means + HDI)
3. Post-period gap plot with 94% HDI (from Bayesian SC)
4. Spaghetti plot with permutation p-value (from Notebook 02)
5. In-time placebo validation
6. Cumulative effect estimate with HDI

### What's Next

You have now completed all three modules of the course:
- **Module 00:** Causal foundations, DAGs, potential outcomes
- **Module 01:** ITS fundamentals and CausalPy API
- **Module 02:** Bayesian ITS with PyMC
- **Module 03:** Synthetic control methods and inference

See the **quick-starts** in the `quick-starts/` directory for ready-to-run templates.

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])